# **10 Mixed domain finetune**

In [ ]:
# ============================================================
#  CELL 4b (NEW) — Fine-tune on Mixed Burst Durations (v2)
#
#  WHERE THIS GOES: paste as a new cell AFTER Cell 4 (the original
#  fine-tuning cell) and AFTER Cell 9 (Financial1.spc test), since
#  it reuses logic from both and needs their saved files:
#    - lstm_model_weights.pt, data_scaler.pkl        (from Cell 2)
#    - Financial1.spc already downloaded              (from Cell 9)
#  It does NOT overwrite anything — saves new files with a "_v2"
#  suffix, so your original fine-tuned model (v1) stays intact for
#  comparison.
#
#  WHAT THIS TESTS: Section V-D found the v1 model (fine-tuned only
#  on short, rare bursts) underperforms reactive scaling on a real
#  trace with sustained elevated load. This cell fine-tunes a NEW
#  model (v2) on a mix of short bursts (original design, 6-16 steps)
#  AND long bursts (60-180 steps, ~34% trace coverage — chosen to
#  roughly match Financial1.spc's measured 33.2% elevated-load
#  fraction from Section V-D). It then evaluates v2 on BOTH the
#  original 30-trial short-burst Monte Carlo test AND the real
#  Financial1.spc trace, so you can see whether the fix helps on
#  real data without silently breaking the original result.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from scipy import stats
import math, pickle, os, bz2, warnings, random
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Cell 4b loaded | Device: {DEVICE}")

WINDOW_SIZE = 60
HORIZON = 3
REQUESTS_PER_SERVER = 150
MIN_SERVERS = 2
MAX_SERVERS = 20
COLD_START_STEPS = 6
STEP_SECONDS = 5

# ── Step 1: mixed-duration workload generator ───────────────────
# Short bursts: same as the original generator (6-16 steps, 500-1200
# intensity). Long bursts: NEW — 60-180 steps (5-15 min), lower
# intensity (300-700) since they're sustained rather than sharp.
# At most one long burst per trace, so worst-case coverage stays
# around 34% of a 720-step trace, not the whole trace.
def generate_mixed_burst_workload(seed, total_steps=720):
    rng = np.random.RandomState(seed)
    base = rng.poisson(lam=100, size=total_steps).astype(float)

    n_short = rng.randint(1, 4)
    for _ in range(n_short):
        center = rng.randint(80, total_steps - 80)
        duration = rng.randint(6, 16)
        intensity = rng.uniform(500, 1200)
        start = max(0, center - duration // 2)
        end = min(total_steps, start + duration)
        base[start:end] += rng.poisson(lam=intensity, size=end - start)

    has_long_burst = rng.random() < 0.5
    if has_long_burst:
        duration = rng.randint(60, 180)
        intensity = rng.uniform(300, 700)
        center = rng.randint(total_steps // 4, 3 * total_steps // 4)
        start = max(0, center - duration // 2)
        end = min(total_steps, start + duration)
        base[start:end] += rng.poisson(lam=intensity, size=end - start)

    return np.clip(base, 20, 3000)

# ── Step 2: load the ORIGINAL pretrained model + scaler (Cell 2) ─
class LSTMPredictor(nn.Module):
    def __init__(self, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, layers, batch_first=True, dropout=dropout)
        self.fc = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

print("📂 Loading pretrained model from Cell 2 (starting fresh, not from v1 fine-tune)...")
try:
    with open("data_scaler.pkl", "rb") as f:
        scaler_v2 = pickle.load(f)
    model_v2 = LSTMPredictor().to(DEVICE)
    model_v2.load_state_dict(torch.load("lstm_model_weights.pt", map_location=DEVICE))
    print("✅ Pretrained model and scaler loaded.")
except FileNotFoundError as e:
    raise RuntimeError(
        "❌ Could not find lstm_model_weights.pt or data_scaler.pkl.\n"
        "   Run Cell 2 (zenodo_trace_training.py) first."
    ) from e

# ── Step 3: build fine-tuning data with mixed burst durations ──
print("\n🔄 Generating mixed-duration fine-tuning data (60 traces, seeds 500-559)...")
finetune_traces = [generate_mixed_burst_workload(s) for s in range(500, 560)]
finetune_data = np.concatenate(finetune_traces)
long_burst_coverage = (finetune_data > 300).mean() * 100
print(f"   Approx. elevated-load coverage in this fine-tuning data: "
      f"{long_burst_coverage:.1f}% of steps above 300 req/s")

real_data_min = scaler_v2.data_min_[0]
real_data_max = scaler_v2.data_max_[0]
spike_data_max = finetune_data.max()
if spike_data_max > real_data_max * 1.5:
    combined_range = np.array([[min(real_data_min, finetune_data.min())],
                                [max(real_data_max, spike_data_max)]])
    scaler_v2 = MinMaxScaler(feature_range=(0, 1))
    scaler_v2.fit(combined_range)
    print(f"   ✅ Scaler refit on range: {scaler_v2.data_min_[0]:.1f} to {scaler_v2.data_max_[0]:.1f}")

finetune_scaled = scaler_v2.transform(finetune_data.reshape(-1, 1)).flatten()
X_ft, y_ft = [], []
for i in range(len(finetune_scaled) - WINDOW_SIZE - HORIZON):
    X_ft.append(finetune_scaled[i:i + WINDOW_SIZE])
    y_ft.append(finetune_scaled[i + WINDOW_SIZE + HORIZON - 1])
X_ft, y_ft = np.array(X_ft), np.array(y_ft)

n = len(X_ft)
train_end = int(n * 0.85)
X_train, y_train = X_ft[:train_end], y_ft[:train_end]
X_val, y_val = X_ft[train_end:], y_ft[train_end:]
print(f"✅ Fine-tuning dataset: {len(X_train)} train | {len(X_val)} val")

# ── Step 4: fine-tune (same hyperparameters as v1, for fair comparison) ─
class LoadDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X).unsqueeze(-1)
        self.y = torch.FloatTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(LoadDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader = DataLoader(LoadDataset(X_val, y_val), batch_size=64, shuffle=False)

FT_EPOCHS, FT_LR = 40, 0.0006
criterion = nn.MSELoss()
optimiser = torch.optim.Adam(model_v2.parameters(), lr=FT_LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser, mode='min', factor=0.5, patience=4)

best_val, best_state = float('inf'), None
print(f"\n🔄 Fine-tuning v2 for {FT_EPOCHS} epochs (lr={FT_LR})...")
for epoch in range(1, FT_EPOCHS + 1):
    model_v2.train()
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        optimiser.zero_grad()
        loss = criterion(model_v2(Xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_v2.parameters(), 1.0)
        optimiser.step()
    model_v2.eval()
    vl_list = []
    with torch.no_grad():
        for Xb, yb in val_loader:
            vl_list.append(criterion(model_v2(Xb.to(DEVICE)), yb.to(DEVICE)).item())
    vl = np.mean(vl_list)
    if vl < best_val:
        best_val = vl
        best_state = {k: v.clone() for k, v in model_v2.state_dict().items()}
    scheduler.step(vl)
    if epoch % 10 == 0 or epoch == 1:
        print(f"  Epoch {epoch:2d}/{FT_EPOCHS} | Val loss: {vl:.6f}")

model_v2.load_state_dict(best_state)
model_v2.eval()
print(f"✅ v2 fine-tuning complete. Best val loss: {best_val:.6f}")

torch.save(model_v2.state_dict(), "lstm_model_finetuned_v2.pt")
with open("data_scaler_finetuned_v2.pkl", "wb") as f:
    pickle.dump(scaler_v2, f)
print("✅ Saved lstm_model_finetuned_v2.pt and data_scaler_finetuned_v2.pkl")
print("   (v1 files are untouched)")

# ── Shared simulation code (same logic used everywhere else) ────
class ServerPool:
    def __init__(self):
        self.active_servers = MIN_SERVERS
        self.pending_servers = 0
        self.pending_countdown = 0
    def step(self, req):
        if self.pending_countdown > 0:
            self.pending_countdown -= 1
            if self.pending_countdown == 0 and self.pending_servers > 0:
                self.active_servers = min(self.active_servers + self.pending_servers, MAX_SERVERS)
                self.pending_servers = 0
        cap = self.active_servers * REQUESTS_PER_SERVER
        util = req / cap
        return util, util > 1.0
    def scale_up(self, n=1):
        if self.active_servers + self.pending_servers < MAX_SERVERS:
            self.pending_servers += n
            self.pending_countdown = COLD_START_STEPS
    def scale_down(self, n=1):
        self.active_servers = max(MIN_SERVERS, self.active_servers - n)

def run_reactive(workload):
    pool, results = ServerPool(), []
    for req in workload:
        util, sla = pool.step(req)
        if util > 0.75: pool.scale_up(1)
        elif util < 0.25 and pool.active_servers > MIN_SERVERS: pool.scale_down(1)
        results.append(sla)
    return sum(results) / len(results) * 100

def run_lstm(workload, model, scaler):
    pool, results = ServerPool(), []
    wl_s = scaler.transform(np.array(workload).reshape(-1, 1)).flatten()
    for t, req in enumerate(workload):
        util, sla = pool.step(req)
        if t >= WINDOW_SIZE:
            window = wl_s[t - WINDOW_SIZE:t]
            x_t = torch.FloatTensor(window).unsqueeze(0).unsqueeze(-1).to(DEVICE)
            with torch.no_grad():
                pred_s = model(x_t).item()
            pred_load = scaler.inverse_transform([[pred_s]])[0][0]
            pred_util = pred_load / (pool.active_servers * REQUESTS_PER_SERVER)
            if pred_util > 0.60:
                needed = math.ceil(pred_load / REQUESTS_PER_SERVER)
                gap = needed - pool.active_servers - pool.pending_servers
                if gap > 0: pool.scale_up(min(gap, 3))
            elif pred_util < 0.25 and pool.active_servers > MIN_SERVERS:
                pool.scale_down(1)
        else:
            if util > 0.75: pool.scale_up(1)
        results.append(sla)
    return sum(results) / len(results) * 100

# ── Test A: original 30-trial SHORT-burst Monte Carlo (unchanged
# generator, seeds 100-129) — checks v2 didn't regress on the
# original test it needs to still pass ─────────────────────────
def generate_short_burst_workload(seed, total_steps=720):
    rng = np.random.RandomState(seed)
    base = rng.poisson(lam=100, size=total_steps).astype(float)
    n_spikes = rng.randint(2, 6)
    for _ in range(n_spikes):
        center = rng.randint(80, total_steps - 80)
        duration = rng.randint(6, 16)
        intensity = rng.uniform(500, 1200)
        start = max(0, center - duration // 2)
        end = min(total_steps, start + duration)
        base[start:end] += rng.poisson(lam=intensity, size=end - start)
    return np.clip(base, 20, 3000)

print("\n🔄 Test A: v2 on the original 30-trial short-burst Monte Carlo...")
v2_short_sla = []
for seed in range(100, 130):
    wl = generate_short_burst_workload(seed)
    v2_short_sla.append(run_lstm(wl, model_v2, scaler_v2))
v2_short_mean = np.mean(v2_short_sla)
print(f"   v2 mean SLA on short bursts: {v2_short_mean:.2f}%")
print(f"   (v1's result on this same test, from Cell 5, was 3.81%)")

# ── Test B: Financial1.spc real trace (same calibration as Cell 9) ─
print("\n🔄 Test B: v2 on the real Financial1.spc trace...")
LOCAL_RAW = "Financial1.spc"
if not os.path.exists(LOCAL_RAW):
    raise RuntimeError(
        "❌ Financial1.spc not found on disk.\n"
        "   Run Cell 9 first (it downloads and decompresses the trace)."
    )

bin_counts = {}
max_bin = 0
with open(LOCAL_RAW, "r", errors="ignore") as f:
    for line in f:
        parts = line.strip().split(",")
        if len(parts) < 5:
            continue
        try:
            ts = float(parts[4])
        except ValueError:
            continue
        b = int(ts // STEP_SECONDS)
        bin_counts[b] = bin_counts.get(b, 0) + 1
        if b > max_bin: max_bin = b
raw_series = np.zeros(max_bin + 1)
for b, c in bin_counts.items():
    raw_series[b] = c

TARGET_MEAN, TARGET_PEAK_DEVIATION = 100.0, 800.0
raw_mean = raw_series.mean()
raw_p99 = np.percentile(raw_series, 99)
k = TARGET_PEAK_DEVIATION / max(raw_p99 - raw_mean, 1e-6)
real_workload = np.clip(TARGET_MEAN + (raw_series - raw_mean) * k, 20, 3000)

v2_real_sla = run_lstm(real_workload, model_v2, scaler_v2)
reactive_real_sla = run_reactive(real_workload)
print(f"   Reactive on real trace: {reactive_real_sla:.2f}%")
print(f"   v2 LSTM on real trace : {v2_real_sla:.2f}%")
print(f"   (v1's result on this same real trace, from Cell 9, was 23.62%)")

# ── Summary ──────────────────────────────────────────────────
print(f"\n{'='*60}")
print("  SUMMARY: v1 (short-bursts-only) vs v2 (mixed durations)")
print(f"{'='*60}")
print(f"  {'Test':<35}{'v1':>10}{'v2':>10}")
print(f"  {'-'*55}")
print(f"  {'Short-burst Monte Carlo (mean SLA%)':<35}{'3.81%':>10}{v2_short_mean:>9.2f}%")
print(f"  {'Financial1.spc real trace (SLA%)':<35}{'23.62%':>10}{v2_real_sla:>9.2f}%")
print(f"  {'Reactive baseline on real trace':<35}{'12.43%':>10}{reactive_real_sla:>9.2f}%")
print(f"{'='*60}")
if v2_real_sla < 12.43:
    print("  ✅ v2 BEATS reactive on the real trace — the fix worked.")
elif v2_real_sla < 23.62:
    print("  ⚠️  v2 improved vs v1 but still loses to reactive on the real trace —")
    print("      partial fix. Report both numbers honestly.")
else:
    print("  ❌ v2 did not improve on the real trace. The long-burst fix as")
    print("     designed did not resolve the gap — also a reportable finding.")
if v2_short_mean > 3.81 + 1.0:
    print("  ⚠️  Note: v2 is noticeably worse on short bursts than v1 —")
    print("      a real tradeoff from training on a harder, more varied task.")

pd.DataFrame([{
    "model": "v1_short_only", "short_burst_sla": 3.81, "real_trace_sla": 23.62,
}, {
    "model": "v2_mixed_duration", "short_burst_sla": v2_short_mean, "real_trace_sla": v2_real_sla,
}]).to_csv("v1_vs_v2_comparison.csv", index=False)
print("\n✅ Comparison saved as v1_vs_v2_comparison.csv")

if '_SEEN_DOWNLOADS' not in globals():
    _SEEN_DOWNLOADS = set()
def safe_download(fname):
    if fname in _SEEN_DOWNLOADS:
        print(f"   (skip {fname} — already downloaded earlier this session)")
        return
    try:
        from google.colab import files
        files.download(fname)
        _SEEN_DOWNLOADS.add(fname)
    except ImportError:
        print(f"   ✅ Saved to disk: {os.path.abspath(fname)}")
        _SEEN_DOWNLOADS.add(fname)
    except Exception as e:
        print(f"   (could not download {fname}: {e})")
for f in ["lstm_model_finetuned_v2.pt", "data_scaler_finetuned_v2.pkl", "v1_vs_v2_comparison.csv"]:
    safe_download(f)

✅ Cell 4b loaded | Device: cuda
📂 Loading pretrained model from Cell 2 (starting fresh, not from v1 fine-tune)...
✅ Pretrained model and scaler loaded.

🔄 Generating mixed-duration fine-tuning data (60 traces, seeds 500-559)...
   Approx. elevated-load coverage in this fine-tuning data: 12.2% of steps above 300 req/s
   ✅ Scaler refit on range: 63.0 to 2047.0
✅ Fine-tuning dataset: 36666 train | 6471 val

🔄 Fine-tuning v2 for 40 epochs (lr=0.0006)...
  Epoch  1/40 | Val loss: 0.003741
  Epoch 10/40 | Val loss: 0.003364
  Epoch 20/40 | Val loss: 0.003315
  Epoch 30/40 | Val loss: 0.003292
  Epoch 40/40 | Val loss: 0.003295
✅ v2 fine-tuning complete. Best val loss: 0.003256
✅ Saved lstm_model_finetuned_v2.pt and data_scaler_finetuned_v2.pkl
   (v1 files are untouched)

🔄 Test A: v2 on the original 30-trial short-burst Monte Carlo...
   v2 mean SLA on short bursts: 3.85%
   (v1's result on this same test, from Cell 5, was 3.81%)

🔄 Test B: v2 on the real Financial1.spc trace...
   Reactiv

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>